# 第 7 週 Part B：診所掛號系統（個人作品）

**規則：** 個人獨立完成；可以使用任何 AI 工具；**禁止與其他同學討論**。

**流程（照順序做）：**
1. 在下一格寫好**規格書**（可以請 AI 幫忙寫初稿，但你要讀過並修改成你要的樣子）。
2. 把規格書交給 AI，請它依照規格產生程式，貼到下面對應的格子。
3. 執行**測試清單**，看懂每一句的 AI 原始回覆與系統解讀。
4. 照最後一格的**交件前檢查清單**交件。

**系統需求（簡化版）：** 科別只有內科、外科、牙科，每科每天有固定名額。功能有掛號、取消、查詢剩餘名額。
🔴 **安全規定：系統不可以提供任何醫療建議或根據症狀推薦科別。** 使用者沒說出科別時，請他直接說要掛哪一科，或洽詢櫃台。

# 📋 規格書（請直接在這一格修改）

學號：411312121 姓名：王奕翔

## 1. 系統目的（一句話）
讓民眾用一句話完成診所的掛號、取消與名額查詢，且系統不提供任何醫療建議。

## 2. 使用者會怎麼說（至少 5 句）
1. 我想掛內科
2. 麻煩幫我預約外科
3. 幫我取消牙科的掛號
4. 牙科今天還有名額嗎
5. 我最近一直頭痛，應該掛哪一科？

## 3. 資料表（最多 2 張）
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| departments | name | TEXT(主鍵) | 科別名稱：內科、外科、牙科 |
| departments | daily_quota | INTEGER | 每日固定名額 |
| appointments | id | INTEGER(自動遞增) | 掛號編號 |
| appointments | dept | TEXT | 掛號的科別 |
| appointments | patient | TEXT | 掛號人姓名 |
| appointments | status | TEXT | 有效/已取消 |
| appointments | created_time | TIMESTAMP | 掛號時間(自動填入) |

初始資料：
- departments：內科 3 名、外科 2 名、牙科 2 名
- appointments：外科已有 陳小華、林小美(外科額滿，方便測試)；牙科已有 王小明(剩1名)；內科沒有人
- 剩餘名額 = 每日名額 - 該科有效的掛號筆數；取消只把狀態改成已取消，不刪除紀錄
- 為了簡化，系統只處理今天的名額，不處理日期

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,科別`(例：`掛號,內科`)
- 動作只能是：掛號、取消、查詢
- 對象只能是：內科、外科、牙科
- 無法判斷時輸出：`未知,未知`
- AI 不可以做的事：提供任何醫療建議、根據症狀推薦科別、自己決定名額、直接操作資料庫

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 掛號 | 科別存在、有名額、此人沒掛過該科 | 掛號成功，並告知剩餘名額 | appointments 新增一筆(狀態＝有效) |
| 掛號 | 該科名額已滿 | 告知名額已滿 | 無 |
| 掛號 | 此人已掛過該科(有效) | 告知不需重複掛號 | 無 |
| 取消 | 找得到此人在該科的有效掛號 | 取消成功 | 該筆狀態改為已取消 |
| 取消 | 找不到有效掛號 | 告知找不到掛號紀錄 | 無 |
| 查詢 | 有說科別 | 告知該科剩餘名額 | 無 |
| 查詢 | 沒說科別 | 列出三科剩餘名額 | 無 |
| 任何功能 | 只有症狀、沒有科別 | 說明系統無法提供醫療建議，請直接說科別或洽詢櫃台 | 無 |

## 6. 例外情況（至少 3 個）
1. 使用者要掛診所沒有的科別(例如眼科)：AI判斷為未知，系統回覆目前只有內科、外科、牙科
2. 使用者只描述症狀(例如：我胸口悶)：不推薦科別、不給醫療建議，請他直接說科別或洽詢櫃台
3. 掛號或取消時沒有輸入姓名：請先輸入姓名，不呼叫資料庫
4. AI回覆格式亂掉、多了符號，或呼叫AI失敗：系統自己從回覆中找出允許的值，找不到就當作未知，程式不當掉
5. 一句話裡有兩個科別(例如：掛內科跟牙科)：只處理AI判斷出的第一個科別

## 7. 測試清單（至少 8 句，正常／錯誤／邊界都要有）
| # | 類型 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 正常 | 我想掛內科 | 掛號,內科 |
| 2 | 正常 | 麻煩幫我預約外科 | 掛號,外科 |
| 3 | 正常 | 我要取消外科的掛號 | 取消,外科 |
| 4 | 正常 | 幫我查一下牙科還有沒有名額 | 查詢,牙科 |
| 5 | 錯誤 | 我要掛眼科 | 科別=未知(系統沒有眼科) |
| 6 | 錯誤 | 我最近一直頭痛，應該掛哪一科？ | 科別=未知(不推薦科別) |
| 7 | 錯誤 | 我胸口悶，幫我掛號 | 科別=未知(不推薦科別) |
| 8 | 錯誤 | 今天天氣如何 | 未知,未知 |
| 9 | 邊界 | 內科 | 科別=內科(動作不限，觀察用) |
| 10 | 邊界 | 我想掛內科跟牙科 | 一句話兩個科別(觀察用) |

## 🤖 可以使用的 prompt

**① 請 AI 寫規格書初稿**
```
我要做一個「診所掛號系統」的課堂小作品（Google Colab + SQLite + Groq + Gradio）。
請依照下面的規格範本，幫我寫出一份規格書初稿。要求：
1. 科別只有：內科、外科、牙科，每科每天有固定名額。
2. 功能：掛號、取消掛號、查詢某科剩餘名額。
3. 資料表不超過 2 張。
4. AI 只負責判斷「動作」與「科別」。如果使用者只描述症狀、沒有說出科別，AI 一律回答未知；系統絕對不可以提供任何醫療建議或推薦科別。
5. 測試清單至少 8 句，要包含正常、錯誤、邊界三種情況，並寫出預期結果。
規格範本如下：
（貼上規格範本）
```

**② 請 AI 依規格寫程式**
```
你是 Python 助教。請依照下面的規格，幫我寫一份 Google Colab 程式，並遵守以下要求：
1. 第 1 格（安裝套件與讀取金鑰）我已經有了，請從第 2 格開始寫，依序為：(2) 建立資料庫與初始資料 (3) 顯示資料表 (4) AI 解讀函式 (5) 測試清單 (6) 功能函式 (7) 主流程與完整流程測試 (8) Gradio 網頁。每格分開給我，開頭用註解說明這格在做什麼。
2. 第 1 格已經定義好 client（Groq）與 MODEL，也已經 import 了 sqlite3、pandas as pd、display、gradio as gr，請直接使用，不要重新讀取金鑰，也不要把金鑰寫在程式裡。
3. 呼叫 Groq 時使用 temperature=0，並用 try/except 包起來，失敗時回傳「未知」，不可讓程式當掉。
4. 每次執行都要先 DROP TABLE IF EXISTS，再重建資料表並放入初始資料。
5. 所有 SQL 都用 ? 參數，不可以用 f-string 組 SQL。
6. 不可直接相信 AI 的回覆：要從回覆中找出允許清單裡的值，找不到就當作「未知」。
7. fetchone() 之後一定要先檢查是否為 None。
8. 資料庫連線用 try/finally 確保關閉。
9. 測試清單的格式為 (使用者說的話, 預期動作, 預期科別)，跑完要印出 AI 原始回覆、系統解讀，以及通過幾句。
10. 只用 sqlite3、groq、gradio、pandas，不要用其他套件。註解用繁體中文，寫得讓初學者看得懂。

規格如下：
（貼上你的規格書）
```

**③ 出錯時請 AI 協助除錯**
```
我的 Colab 程式出現以下錯誤。請先用兩三句話解釋原因，再告訴我「只需要修改哪幾行」，不要整份重寫。
【錯誤訊息】
（貼上完整錯誤訊息）
【出錯的那一格程式】
（貼上程式）
```

In [ ]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型
# ============================================================
!pip install -q groq gradio

import sqlite3
import pandas as pd
from IPython.display import display
from google.colab import userdata
from groq import Groq
import gradio as gr

# 🔴 金鑰只能從左側「🔑 密鑰（Secrets）」讀取，絕對不可以寫成 api_key="gsk_..."
#    寫死的金鑰一旦存到 GitHub，就等於公開給全世界使用。
#    也不要 print 出金鑰，因為輸出結果也會一起存到 GitHub。
try:
    client = Groq(api_key=userdata.get('SDDA_Lecture06'))
except Exception as e:
    print("❌ 讀不到金鑰。請檢查：")
    print("   1. 左側 🔑 是否新增了名稱為 SDDA_Lecture06 的密鑰（大小寫要完全一樣）")
    print("   2. 該密鑰的「筆記本存取權」開關是否已打開")
    raise

# 模型名稱集中放在這裡；日後模型更換或下架，只要改這一行
MODEL = "openai/gpt-oss-120b"
print("✅ Groq 金鑰讀取成功，使用模型：", MODEL)

✅ Groq 金鑰讀取成功，使用模型： openai/gpt-oss-120b


In [ ]:
# ============================================================
# 第 2 格：建立資料庫與初始資料（每次執行都會重建，確保「從頭跑」結果一致）
# ============================================================
DB = "clinic.db"
DEPTS = ["內科", "外科", "牙科"]   # 系統認得的科別（AI 的答案只能是這些）
ACTIVE = "有效"                     # ⭐ 狀態文字集中定義，避免各處打錯字
CANCELLED = "已取消"

conn = sqlite3.connect(DB)
try:
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS appointments")
    cursor.execute("DROP TABLE IF EXISTS departments")

    cursor.execute("""
        CREATE TABLE departments (
            name        TEXT PRIMARY KEY,     -- 科別名稱
            daily_quota INTEGER NOT NULL      -- 每日固定名額
        )
    """)
    cursor.execute("""
        CREATE TABLE appointments (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            dept         TEXT NOT NULL,       -- 掛號的科別
            patient      TEXT NOT NULL,       -- 掛號人姓名
            status       TEXT NOT NULL,       -- 有效 / 已取消
            created_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    cursor.executemany(
        "INSERT INTO departments (name, daily_quota) VALUES (?, ?)",
        [("內科", 3), ("外科", 2), ("牙科", 2)]
    )
    # 外科已額滿、牙科剩 1 名，方便測試「額滿」與「取消後釋出名額」
    cursor.executemany(
        "INSERT INTO appointments (dept, patient, status) VALUES (?, ?, ?)",
        [("外科", "陳小華", ACTIVE), ("外科", "林小美", ACTIVE), ("牙科", "王小明", ACTIVE)]
    )
    conn.commit()
finally:
    conn.close()                      # ⭐ 不管有沒有出錯，連線一定會關閉
print("✅ 資料庫建立完成")

✅ 資料庫建立完成


In [ ]:
# ============================================================
# 第 3 格：顯示資料表（之後每次操作完都可以呼叫，確認資料庫真的有變）
# ============================================================
def show_tables():
    conn = sqlite3.connect(DB)
    try:
        print("📦 各科剩餘名額（剩餘 ＝ 每日名額 − 有效掛號數）")
        sql = """
            SELECT d.name AS 科別,
                   d.daily_quota AS 每日名額,
                   COUNT(a.id) AS 已掛號,
                   d.daily_quota - COUNT(a.id) AS 剩餘
            FROM departments d
            LEFT JOIN appointments a ON a.dept = d.name AND a.status = ?
            GROUP BY d.name, d.daily_quota
        """
        display(pd.read_sql_query(sql, conn, params=(ACTIVE,)))   # ⭐ 用 ? 參數，不用 f-string 組 SQL
        print("📝 appointments（掛號紀錄）")
        display(pd.read_sql_query("SELECT * FROM appointments", conn))
    finally:
        conn.close()

show_tables()

📦 各科剩餘名額（剩餘 ＝ 每日名額 − 有效掛號數）


,科別,每日名額,已掛號,剩餘
0,內科,3,0,3
1,外科,2,2,0
2,牙科,2,1,1


📝 appointments（掛號紀錄）


,id,dept,patient,status,created_time
0,1,外科,陳小華,有效,2026-10-27 07:01:15
1,2,外科,林小美,有效,2026-10-27 07:01:15
2,3,牙科,王小明,有效,2026-10-27 07:01:15


In [ ]:
# ============================================================
# 第 4 格：AI 解讀使用者的話 → (動作, 科別, 原始回覆)
# ============================================================
# ⭐ 順序很重要：「取消掛號」同時含有「掛號」兩個字，所以要先檢查「取消」，再檢查「掛號」
ACTIONS = ["取消", "查詢", "掛號"]
dept_list = "、".join(DEPTS)

def ai_parse(user_say):
    prompt = f"""你是診所的掛號櫃台助理，只負責判斷「動作」與「科別」。
診所只有這些科別：{dept_list}。
請判斷使用者想「掛號」、「取消」還是「查詢」剩餘名額，以及使用者「明確說出」的科別。

只能回覆一行，格式為：動作,科別
- 動作只能是：掛號、取消、查詢、未知
- 科別只能是：{dept_list}、未知
- 使用者只描述症狀、沒有說出科別時，科別一律回答：未知
- 絕對不可以提供醫療建議，也不可以根據症狀推薦科別
- 不要回覆任何其他文字

範例：
使用者：我想掛內科 → 掛號,內科
使用者：幫我取消外科的掛號 → 取消,外科
使用者：牙科今天還有名額嗎 → 查詢,牙科
使用者：我最近一直頭痛 → 未知,未知
使用者：我胸口悶想掛號，應該看哪一科 → 掛號,未知
使用者：今天天氣如何 → 未知,未知

使用者：{user_say}"""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,                               # ⭐ 溫度 0：同一句話盡量得到同樣的答案
        )
        raw = response.choices[0].message.content or ""  # ⭐ content 偶爾會是 None
    except Exception as e:
        # ⭐ 網路不穩、金鑰錯誤、超過使用量時，不要讓整個程式當掉
        print("⚠️ 呼叫 AI 失敗：", e)
        return "未知", "未知", ""

    # ⭐ 不直接相信 AI 的格式，自己從回覆中「找」出允許的答案
    action = "未知"
    for a in ACTIONS:
        if a in raw:
            action = a
            break

    dept = "未知"
    for name in DEPTS:
        if name in raw:
            dept = name
            break

    # ⭐ 安全保護：科別一定要真的出現在使用者原本的話裡
    #    如果 AI 自己從症狀「猜」出科別（使用者沒說），一律當作未知，系統絕不推薦科別
    if dept != "未知" and dept not in user_say:
        dept = "未知"

    return action, dept, raw

In [ ]:
# ============================================================
# 第 5 格：測試清單（與規格書第 7 節一致）
# 每列是 (使用者說的話, 預期動作, 預期科別)；"*" 表示「這一項不在意」
# ============================================================
test_list = [
    ("我想掛內科",                       "掛號", "內科"),   # 1 正常
    ("麻煩幫我預約外科",                 "掛號", "外科"),   # 2 正常：說法不同
    ("我要取消外科的掛號",               "取消", "外科"),   # 3 正常：取消（含「掛號」二字）
    ("幫我查一下牙科還有沒有名額",       "查詢", "牙科"),   # 4 正常：查詢
    ("我要掛眼科",                       "*",    "未知"),   # 5 錯誤：沒有這個科別
    ("我最近一直頭痛，應該掛哪一科？",   "*",    "未知"),   # 6 錯誤：只有症狀，不可推薦科別
    ("我胸口悶，幫我掛號",               "*",    "未知"),   # 7 錯誤：只有症狀，不可推薦科別
    ("今天天氣如何",                     "未知", "未知"),   # 8 錯誤：無關的話
    ("內科",                             "*",    "內科"),   # 9 邊界：只說科別
    ("我想掛內科跟牙科",                 "*",    "*"),      # 10 邊界：一句話兩個科別（觀察用）
]

passed = 0
for say, exp_action, exp_dept in test_list:
    action, dept, raw = ai_parse(say)
    ok = (exp_action in ("*", action)) and (exp_dept in ("*", dept))
    passed += ok
    print(f"{'✅' if ok else '❌'} {say}")
    print(f"     AI 原始回覆：{raw!r}")
    print(f"     系統解讀：{action} / {dept}（預期：{exp_action} / {exp_dept}）\n")

print(f"通過 {passed} / {len(test_list)} 句")

✅ 我想掛內科
     AI 原始回覆：'掛號,內科'
     系統解讀：掛號 / 內科（預期：掛號 / 內科）

✅ 麻煩幫我預約外科
     AI 原始回覆：'掛號,外科'
     系統解讀：掛號 / 外科（預期：掛號 / 外科）

✅ 我要取消外科的掛號
     AI 原始回覆：'取消,外科'
     系統解讀：取消 / 外科（預期：取消 / 外科）

✅ 幫我查一下牙科還有沒有名額
     AI 原始回覆：'查詢,牙科'
     系統解讀：查詢 / 牙科（預期：查詢 / 牙科）

✅ 我要掛眼科
     AI 原始回覆：'掛號,未知'
     系統解讀：掛號 / 未知（預期：* / 未知）

✅ 我最近一直頭痛，應該掛哪一科？
     AI 原始回覆：'掛號,未知'
     系統解讀：掛號 / 未知（預期：* / 未知）

✅ 我胸口悶，幫我掛號
     AI 原始回覆：'掛號,未知'
     系統解讀：掛號 / 未知（預期：* / 未知）

✅ 今天天氣如何
     AI 原始回覆：'未知,未知'
     系統解讀：未知 / 未知（預期：未知 / 未知）

✅ 內科
     AI 原始回覆：'未知,內科'
     系統解讀：未知 / 內科（預期：* / 內科）

✅ 我想掛內科跟牙科
     AI 原始回覆：'掛號,未知'
     系統解讀：掛號 / 未知（預期：* / *）

通過 10 / 10 句


In [ ]:
# ============================================================
# 第 6 格：功能函式（掛號、取消、查詢）— 直接操作資料庫，不呼叫 AI
# ============================================================
def get_quota(cursor, dept):
    """查某科的每日名額；查不到科別回傳 None"""
    cursor.execute("SELECT daily_quota FROM departments WHERE name = ?", (dept,))
    row = cursor.fetchone()
    if row is None:                       # ⭐ fetchone() 之後一定先檢查 None
        return None
    return row[0]

def count_active(cursor, dept):
    """某科目前「有效」的掛號人數"""
    cursor.execute("SELECT COUNT(*) FROM appointments WHERE dept = ? AND status = ?", (dept, ACTIVE))
    row = cursor.fetchone()
    return row[0] if row is not None else 0


def register(dept, patient):
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()
        quota = get_quota(cursor, dept)
        if quota is None:
            return f"系統沒有「{dept}」這個科別。"

        # 同一個人不能重複掛同一科（只看狀態為「有效」的）
        cursor.execute(
            "SELECT id FROM appointments WHERE dept = ? AND patient = ? AND status = ?",
            (dept, patient, ACTIVE)
        )
        if cursor.fetchone() is not None:
            return f"{patient} 已經掛過「{dept}」了，不需要重複掛號。"

        used = count_active(cursor, dept)
        if used >= quota:
            return f"不好意思，「{dept}」今天的名額已滿，請改掛其他科別或洽詢櫃台。"

        cursor.execute(
            "INSERT INTO appointments (dept, patient, status) VALUES (?, ?, ?)",
            (dept, patient, ACTIVE)
        )
        conn.commit()
        return f"🎉 {patient} 掛號「{dept}」成功！該科今天還剩 {quota - used - 1} 個名額。"
    finally:
        conn.close()                      # ⭐ 不管從哪一行 return，連線一定會關閉


def cancel(dept, patient):
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()
        if get_quota(cursor, dept) is None:
            return f"系統沒有「{dept}」這個科別。"

        # 找出這個人在這一科「最新一筆、還有效」的掛號
        cursor.execute(
            "SELECT MAX(id) FROM appointments WHERE dept = ? AND patient = ? AND status = ?",
            (dept, patient, ACTIVE)
        )
        row = cursor.fetchone()
        if row is None or row[0] is None:  # ⭐ MAX() 找不到資料時，row 存在但 row[0] 是 None
            return f"找不到 {patient} 在「{dept}」的有效掛號紀錄，無法取消。"

        cursor.execute("UPDATE appointments SET status = ? WHERE id = ?", (CANCELLED, row[0]))
        conn.commit()
        return f"✅ 已取消 {patient} 在「{dept}」的掛號。"
    finally:
        conn.close()


def query_remaining(dept):
    """查詢某科剩餘名額；科別是「未知」時，列出所有科別"""
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()
        targets = DEPTS if dept == "未知" else [dept]
        results = []
        for d in targets:
            quota = get_quota(cursor, d)
            if quota is None:
                return f"系統沒有「{d}」這個科別。"
            results.append(f"{d}剩 {quota - count_active(cursor, d)} 個（名額 {quota}）")
        return "📋 今日剩餘名額：" + "；".join(results)
    finally:
        conn.close()

In [ ]:
# ============================================================
# 第 7 格：主流程 —— 檢查輸入 → AI 解讀 → 掛號／取消／查詢；最後測完整流程
# ============================================================
HELP_DEPT = f"請直接告訴我要掛哪一科（{dept_list}），或洽詢櫃台。"

def clinic_system(user_say, patient):
    # ⭐ 先檢查輸入，空白的就不要浪費一次 AI 呼叫
    if not user_say or not user_say.strip():
        return "請輸入你想掛號、取消或查詢名額的內容。"
    patient = (patient or "").strip()

    action, dept, raw = ai_parse(user_say)
    print(f"📌 [AI 解讀] 動作={action}，科別={dept}（原始回覆：{raw!r}）")

    if action == "查詢":
        return query_remaining(dept)              # 查詢不需要姓名；沒說科別就列出全部

    if action in ("掛號", "取消"):
        if dept == "未知":
            # 🔴 安全規定：不提供醫療建議、不推薦科別
            return "系統無法提供醫療建議或推薦科別。" + HELP_DEPT
        if not patient:
            return "請先輸入你的姓名。"
        if action == "掛號":
            return register(dept, patient)
        return cancel(dept, patient)

    # 動作是「未知」
    if dept == "未知":
        return "抱歉，我只能處理掛號、取消與查詢名額，無法提供醫療建議或推薦科別。" + HELP_DEPT
    return "抱歉，我不確定你要掛號、取消還是查詢，請再說清楚一點。"


# ---------- 完整流程測試（請對照每一行後面的「預期」）----------
cases = [
    ("我想掛內科",                   "王小明"),   # 預期：掛號成功（內科剩 2）
    ("我想掛內科",                   "王小明"),   # 預期：已經掛過，不需重複
    ("我要掛外科",                   "陳大文"),   # 預期：外科名額已滿
    ("我要取消外科的掛號",           "陳小華"),   # 預期：取消成功（釋出 1 個名額）
    ("我要掛外科",                   "陳大文"),   # 預期：這次掛號成功
    ("幫我查一下牙科還有沒有名額",   ""),         # 預期：牙科剩 1 個（查詢不用姓名）
    ("我要取消內科的掛號",           "李四"),     # 預期：找不到掛號紀錄
    ("我最近一直頭痛，要掛哪一科比較好？", "林小美"),  # 預期：不給建議，請直接說科別
    ("我要掛內科",                   ""),         # 預期：請先輸入姓名
]
for say, name in cases:
    print("-" * 50)
    print(f"🙋 {name or '（未填姓名）'}：{say}")
    print(clinic_system(say, name))

print("-" * 50)
show_tables()

--------------------------------------------------
🙋 王小明：我想掛內科
📌 [AI 解讀] 動作=掛號，科別=內科（原始回覆：'掛號,內科'）
🎉 王小明 掛號「內科」成功！該科今天還剩 2 個名額。
--------------------------------------------------
🙋 王小明：我想掛內科
📌 [AI 解讀] 動作=掛號，科別=內科（原始回覆：'掛號,內科'）
王小明 已經掛過「內科」了，不需要重複掛號。
--------------------------------------------------
🙋 陳大文：我要掛外科
📌 [AI 解讀] 動作=掛號，科別=外科（原始回覆：'掛號,外科'）
不好意思，「外科」今天的名額已滿，請改掛其他科別或洽詢櫃台。
--------------------------------------------------
🙋 陳小華：我要取消外科的掛號
📌 [AI 解讀] 動作=取消，科別=外科（原始回覆：'取消,外科'）
✅ 已取消 陳小華 在「外科」的掛號。
--------------------------------------------------
🙋 陳大文：我要掛外科
📌 [AI 解讀] 動作=掛號，科別=外科（原始回覆：'掛號,外科'）
🎉 陳大文 掛號「外科」成功！該科今天還剩 0 個名額。
--------------------------------------------------
🙋 （未填姓名）：幫我查一下牙科還有沒有名額
📌 [AI 解讀] 動作=查詢，科別=牙科（原始回覆：'查詢,牙科'）
📋 今日剩餘名額：牙科剩 1 個（名額 2）
--------------------------------------------------
🙋 李四：我要取消內科的掛號
📌 [AI 解讀] 動作=取消，科別=內科（原始回覆：'取消,內科'）
找不到 李四 在「內科」的有效掛號紀錄，無法取消。
--------------------------------------------------
🙋 林小美：我最近一直頭痛，要掛哪一科比較好？
📌 [AI 解讀] 動作=掛號，科別=未知（原始回覆：'掛號,未知

,科別,每日名額,已掛號,剩餘
0,內科,3,1,2
1,外科,2,2,0
2,牙科,2,1,1


📝 appointments（掛號紀錄）


,id,dept,patient,status,created_time
0,1,外科,陳小華,已取消,2026-10-27 07:01:15
1,2,外科,林小美,有效,2026-10-27 07:01:15
2,3,牙科,王小明,有效,2026-10-27 07:01:15
3,4,內科,王小明,有效,2026-10-27 07:01:26
4,5,外科,陳大文,有效,2026-10-27 07:01:28


In [ ]:
# ============================================================
# 第 8 格：Gradio 網頁（放在最後一格）
# 提醒：share 連結一段時間後會失效，GitHub 上也看不到網頁；
#       批改看的是第 5、7 格的執行結果，網頁只是讓你自己操作用的。
# ============================================================
demo = gr.Interface(
    fn=clinic_system,                        # 輸入框的順序 = 函式參數的順序
    inputs=[
        gr.Textbox(label="💬 你想做什麼？（例如：我想掛內科）"),
        gr.Textbox(label="🙋 你的姓名（掛號、取消時需要）"),
    ],
    outputs=gr.Textbox(label="🤖 系統回應"),
    title="🏥 診所掛號系統",
    description="可掛號、取消、查詢剩餘名額（內科、外科、牙科）。本系統不提供醫療建議，請直接說出要掛的科別。",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2a38e0b5e67059bdcd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# ✅ 交件前檢查清單（每一項都要打勾）

- [ ] 程式裡**沒有**任何 `gsk_` 開頭的金鑰，也沒有 print 出金鑰
- [ ] 規格書第 1～7 節都填完，測試清單至少 8 句
- [ ] 選單「執行階段 → 重新啟動並執行全部」，**全部跑完沒有紅色錯誤**
- [ ] 每一格左邊的執行編號是**連續遞增**的（1、2、3…）
- [ ] 選單「編輯 → 筆記本設定」，**沒有**勾選「儲存筆記本時忽略程式碼儲存格輸出內容」
- [ ] 選單「檔案 → 在 GitHub 中儲存副本」，存到**公開（Public）**的 repo
- [ ] 用無痕視窗打開 GitHub 上的檔案，確認看得到程式和執行結果